# Anomaly Detection Project - Full EDA and Model Comparison

This notebook demonstrates a complete machine learning workflow for sensor-based anomaly detection using a synthetic but realistic dataset.

It includes:
- loading the dataset
- checking shape and schema
- descriptive statistics
- data quality checks
- EDA and visualization
- model benchmarking
- confusion matrix and interpretation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Load the dataset

This project uses a synthetic sensor dataset with features such as temperature, humidity, pressure, and vibration. It includes a `label` column where `0` = normal and `1` = anomaly.

The dataset is created to mimic real-world IoT / industrial sensor behavior.

In [ ]:
data_path = Path('../data/sensor_anomalies_dataset.csv')
df = pd.read_csv(data_path)
df.head()

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(df.dtypes)
print()
print(df.isnull().sum())

## 2. Data description and summary statistics

We examine the basic statistical distribution of every field and check the class balance.

In [ ]:
df.describe().T

In [ ]:
df['label'].value_counts().rename(index={0: 'normal', 1: 'anomaly'})

In [ ]:
df.groupby('label')[['temperature', 'humidity', 'pressure', 'vibration']].mean()

## 3. Univariate distribution analysis

This helps reveal how normal and anomalous readings separate across sensor variables.

In [ ]:
for col in ['temperature', 'humidity', 'pressure', 'vibration']:
    plt.figure()
    sns.histplot(data=df, x=col, hue='label', kde=True, multiple='layer', alpha=0.6)
    plt.title(f'Distribution of {col} by class')
    plt.show()


## 4. Correlation analysis

We inspect how strongly the features are related to one another.

In [ ]:
corr = df.drop(columns=['label']).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()


## 5. Boxplots to inspect outliers

Anomaly detection depends heavily on unusual values and extreme readings. Boxplots highlight where those outliers appear.

In [ ]:
for col in ['temperature', 'humidity', 'pressure', 'vibration']:
    plt.figure()
    sns.boxplot(data=df, x='label', y=col)
    plt.title(f'{col} by label')
    plt.xticks([0, 1], ['normal', 'anomaly'])
    plt.show()


## 6. Prepare data for modeling

We split the dataset into train and test sets and keep the target labels for evaluation.

In [ ]:
X = df[['temperature', 'humidity', 'pressure', 'vibration']].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

## 7. Define model evaluation utilities


In [ ]:
def to_binary(preds):
    return np.where(np.asarray(preds) == -1, 1, 0)

def evaluate_model(y_true, preds):
    y_pred = to_binary(preds)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }


## 8. Benchmark five anomaly detection models

We compare Isolation Forest, Local Outlier Factor, One-Class SVM, DBSCAN, and Elliptic Envelope.

In [ ]:
models = {
    'Isolation Forest': IsolationForest(contamination=0.08, random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(n_neighbors=20, contamination=0.08),
    'One-Class SVM': OneClassSVM(nu=0.08, kernel='rbf', gamma='scale'),
    'DBSCAN': DBSCAN(eps=1.2, min_samples=15),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.08),
}

results = []
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for name, model in models.items():
    if name == 'Isolation Forest':
        model.fit(X_train_scaled)
        preds = model.predict(X_test_scaled)
    elif name == 'Local Outlier Factor':
        preds = model.fit_predict(X_test_scaled)
    elif name == 'One-Class SVM':
        model.fit(X_train_scaled)
        preds = model.predict(X_test_scaled)
    elif name == 'DBSCAN':
        preds = model.fit_predict(X_test_scaled)
    elif name == 'Elliptic Envelope':
        model.fit(X_train_scaled)
        preds = model.predict(X_test_scaled)

    metrics = evaluate_model(y_test, preds)
    results.append({'model': name, **metrics})
    print(f"{name}: {metrics}")

comparison_df = pd.DataFrame(results).sort_values('f1', ascending=False)
comparison_df

## 9. Visualize model performance


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_df, x='f1', y='model', palette='viridis')
plt.title('Model Comparison by F1 Score')
plt.xlabel('F1 Score')
plt.ylabel('Model')
plt.show()


## 10. Confusion matrix for the best model

This gives a more interpretable view of how many normal and anomalous points were correctly classified.

In [ ]:
best_model_name = comparison_df.iloc[0]['model']
best_model = models[best_model_name]

if best_model_name == 'Isolation Forest':
    best_preds = best_model.predict(X_test_scaled)
elif best_model_name == 'Local Outlier Factor':
    best_preds = best_model.fit_predict(X_test_scaled)
elif best_model_name == 'One-Class SVM':
    best_preds = best_model.predict(X_test_scaled)
elif best_model_name == 'DBSCAN':
    best_preds = best_model.fit_predict(X_test_scaled)
else:
    best_preds = best_model.predict(X_test_scaled)

best_binary = to_binary(best_preds)
cm = confusion_matrix(y_test, best_binary)
print(f'Best model: {best_model_name}')
print(cm)

display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'anomaly'])
display.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()


## 11. Final takeaway

This notebook follows the full machine learning workflow:
- data loading and inspection
- descriptive statistics
- EDA
- algorithm benchmarking
- confusion matrix interpretation
- final model selection

This is exactly the kind of project structure that looks professional in a portfolio, GitHub repo, or interview discussion.